# Phân tích truy vấn TPC-H Q6 và Q17

Thực nghiệm sử dụng DataFusion **32.0.0**, DuckDB **0.9.1** và dữ liệu TPC-H **SF=10** lưu dưới dạng Parquet. Phần chạy ban đầu thực thi mỗi truy vấn một lần trên mỗi bộ máy, DataFusion trước rồi DuckDB.

Riêng Q6 có thêm một lần `EXPLAIN ANALYZE VERBOSE` của DataFusion để thu bộ đếm. DuckDB bật JSON profiling ngay trong lần SELECT ban đầu; sau khi tự sửa cấu hình, chạy thêm một lần đối chứng. Các ô hiển thị ảnh, EXPLAIN và bảng số liệu không tự chạy lại SELECT.

Thời gian ngoài API bao gồm lập kế hoạch, thực thi và thu kết quả. Mỗi cấu hình chỉ đo một lượt, không xóa cache; đây là minh họa trên máy hiện tại, không phải kết quả bài báo hay kết luận về mọi lần chạy.


## Kiến trúc DataFusion và vị trí của truy vấn

![Hình 2 của bài báo, được bổ sung số khoanh tròn từ 1 đến 8 để đối chiếu với mã và kế hoạch](Architecture-DataFusion-Numbered.png)

*Nguồn: Lamb và cộng sự, Hình 2, tr. 8. Các số khoanh tròn do nhóm bổ sung để chú giải; nội dung kiến trúc gốc được giữ nguyên. [Bài báo](https://doi.org/10.1145/3626246.3653368).*

| Số trên hình | Thành phần và công việc | Mã hoặc đầu ra tương ứng |
|---|---|---|
| **1** | **SQL:** nhận câu truy vấn và tạo biểu diễn logic. | `ctx.sql(q6_sql)` hoặc `ctx.sql(q17_sql)` |
| **2** | **Catalog and Data Sources:** biết bảng, kiểu cột và nơi đọc dữ liệu. | `CREATE EXTERNAL TABLE ... STORED AS PARQUET`; nguồn là `part-0.parquet` |
| **3** | **LogicalPlan:** mô tả các phép đọc, lọc, nối và tổng hợp. | `initial_logical_plan` là ban đầu; `logical_plan` là đầu ra cuối của bước tối ưu logic |
| **4** | **Vòng biến đổi logic:** áp dụng các quy tắc phân tích, tối ưu. | `logical_plan after ...`, ví dụ `push_down_filter`, `push_down_projection` |
| **5** | **ExecutionPlan:** cây toán tử vật lý được chọn để thực hiện yêu cầu. | `initial_physical_plan` trước tối ưu vật lý; `physical_plan` là kế hoạch cuối |
| **6** | **Vòng biến đổi vật lý:** điều chỉnh cách thực hiện kế hoạch. | `physical_plan after ...`, ví dụ `join_selection`, `coalesce_batches` |
| **7** | **Execution Engine:** các toán tử đọc, lọc, tính biểu thức, nối và tổng hợp khi chạy. | Q6: `ParquetExec`, `FilterExec`, `AggregateExec`; Q17 còn có các phép nối |
| **8** | **Streams:** các toán tử truyền các lô Arrow khi thực thi. | `ctx.sql(sql).collect()` chạy truy vấn và thu các `RecordBatch` kết quả |

**Luồng lập kế hoạch:** SQL **1** + thông tin bảng **2** → kế hoạch logic **3**, biến đổi tại **4** → kế hoạch vật lý **5**, biến đổi tại **6**.

**Luồng thực thi:** khi gọi `collect()` cho truy vấn thật, các toán tử **7** lấy dữ liệu từ nguồn **2**, xử lý và truyền các lô qua **8** đến kết quả. Hình kiến trúc không phải sơ đồ cho thấy dữ liệu Parquet chạy xuyên qua cây logic.

Các vùng xanh dương là điểm mở rộng. Phần Python gốc của Q6/Q17 dùng thành phần có sẵn; phần Rust Q17 từ mục 2.4 đăng ký quy tắc tối ưu vật lý bổ sung dựa trên thống kê. Số **8** đánh dấu vùng Streams nói chung, không có nghĩa ví dụ đang dùng thành phần Extension Stream.

*Tham chiếu: mục 5.1–5.5, 6.1, 7.6–7.7 của bài báo, tr. 8–12.*


### Q6 và Q17 đi qua kiến trúc như thế nào?

| Truy vấn | Kế hoạch cần biểu diễn và thực thi | Các vùng trên hình |
|---|---|---|
| **Q6** | Đọc `lineitem` → giữ dòng đủ điều kiện → tính và cộng tiền giảm giá → trả một giá trị. | SQL **1**, Parquet **2**, kế hoạch và tối ưu **3–6**, lọc và tổng hợp **7**, lô kết quả **8** |
| **Q17** | Đọc `part` và `lineitem` → lọc mặt hàng → tính số lượng trung bình của từng mặt hàng → nối, lọc các dòng nhỏ → cộng giá trị và chia 7. | Cùng luồng **1–8**; có thêm tổng hợp theo khóa và các phép nối ở **7** |

Bảng mô tả yêu cầu xử lý. Thứ tự và cách kết hợp toán tử cụ thể được xác định bằng kế hoạch thật của từng bộ máy. Trong cây kế hoạch, nhánh nguồn nằm dưới và cung cấp đầu vào cho toán tử phía trên.

Q6 không có khóa `GROUP BY`, nên khối HashAggregate trong hình không có nghĩa Q6 phải tạo bảng băm cho nhiều nhóm. Cả Q6 và Q17 đều không có `ORDER BY`, nên không tự thêm bước Sort chỉ vì sơ đồ có khối đó.

*Tham chiếu: Hình 2, mục 5.1 và 5.5 của [bài báo](https://doi.org/10.1145/3626246.3653368), tr. 8–9.*


### Phiên bản phần mềm trong bài báo

![Đoạn trích bài báo xác định DataFusion 32.0.0 và DuckDB 0.9.1 được sử dụng qua giao diện Python](DataFusion-Environtment.png)

**Bản dịch:**

> Hiệu năng của DataFusion 32.0.0 và DuckDB 0.9.1, hai phiên bản mới nhất tại thời điểm viết bài, được đo thông qua giao diện Python tương ứng. Hiệu quả sử dụng từng lõi được đánh giá trong mục 8.1; khả năng mở rộng khi chạy đa luồng được đánh giá trong mục 8.2. Các tập lệnh thực nghiệm được công bố trực tuyến.

*Nguồn: A. Lamb và cộng sự, mục 8, [Apache Arrow DataFusion: A Fast, Embeddable, Modular Analytic Query Engine](https://doi.org/10.1145/3626246.3653368), 2024.*

Bài báo sử dụng DataFusion **32.0.0** và DuckDB **0.9.1** thông qua giao diện Python. Phần minh họa sử dụng cùng hai phiên bản này.


In [ ]:
# Path biểu diễn đường dẫn tệp và thư mục.
from pathlib import Path
# perf_counter lấy mốc thời gian để tính thời gian chạy.
from time import perf_counter
# re tìm và thay đường dẫn trong chuỗi SQL theo mẫu.
import re
# sys cho biết Python đang chạy từ môi trường nào.
import sys
# math.isclose so sánh kết quả số thực với sai số nhỏ.
import math

# Nạp hai thư viện xử lý truy vấn; chưa thực thi SQL.
import datafusion
# Nạp tiện ích phục vụ phần minh họa, chưa chạy truy vấn khi import.
import duckdb

# In thông tin để phát hiện notebook chọn sai kernel.
print("Python:", sys.executable)
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print("DataFusion:", datafusion.__version__)
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print("DuckDB:", duckdb.__version__)
# Dừng nếu phiên bản không đúng môi trường đang tái hiện.
if datafusion.__version__ != "32.0.0" or duckdb.__version__ != "0.9.1":
    # Dừng với thông báo rõ ràng thay vì tiếp tục bằng dữ liệu hoặc cấu hình không hợp lệ.
    raise RuntimeError("Sai kernel. Chọn Python (TPC-H DataFusion 32) hoặc venv/bin/python của kho benchmark, rồi chạy lại ô này.")

# Các lớp tạo phiên truy vấn và cấu hình DataFusion.
from datafusion import SessionContext, SessionConfig, RuntimeConfig

# Bắt đầu tìm từ thư mục kernel đang làm việc.
current_dir = Path.cwd()
# Chưa tìm thấy thư mục gốc thì để giá trị None.
root = None
# Kiểm tra thư mục hiện tại rồi lần lượt các thư mục cha.
for candidate in (current_dir, *current_dir.parents):
    # Đây phải là kho có cả tập truy vấn và chương trình benchmark.
    if (candidate / "tpch" / "queries.sql").is_file() and (candidate / "common" / "run-query-datafusion.py").is_file():
        # Lưu thư mục gốc tìm được rồi dừng tìm.
        root = candidate
        # Đã tìm thấy mục cần dùng, dừng vòng lặp để tránh đọc thêm.
        break
# Không tìm được kho thì báo lỗi rõ ràng.
if root is None:
    # Dừng với thông báo rõ ràng thay vì tiếp tục bằng dữ liệu hoặc cấu hình không hợp lệ.
    raise RuntimeError("Mở notebook từ thư mục kho benchmark hoặc tpch.")

# Xác định thư mục chứa SQL và thư mục dữ liệu đã tạo.
tpch_dir = root / "tpch"
# Giá trị trung gian `data_dir` được dùng ngay ở bước bên dưới.
data_dir = root / "arrow-datafusion" / "benchmarks" / "data" / "tpch_dataset"
# Danh sách tám bảng đầu vào của TPC-H.
tables = ("customer", "orders", "lineitem", "part", "partsupp", "region", "supplier", "nation")
# Tìm bảng còn thiếu tệp Parquet.
missing = [t for t in tables if not (data_dir / t / "part-0.parquet").is_file()]
# Dừng trước khi đăng ký bảng nếu chưa có dữ liệu.
if missing:
    # Dừng với thông báo rõ ràng thay vì tiếp tục bằng dữ liệu hoặc cấu hình không hợp lệ.
    raise FileNotFoundError(f"Chưa có Parquet cho {missing}; cần chạy setup.sh trước.")

# Đọc tệp SQL và tách theo dòng: mỗi dòng ứng với một số truy vấn.
queries = (tpch_dir / "queries.sql").read_text().splitlines()
# Danh sách Python bắt đầu từ 0 nên Q6 ở vị trí 5.
q6_sql = queries[5]
# Q17 ở vị trí 16.
q17_sql = queries[16]
# In đường dẫn dữ liệu mà notebook sẽ dùng.
print("Dữ liệu:", data_dir)


### Tệp Parquet và các bảng chứa thông tin gì?

TPC-H mô phỏng hoạt động đặt hàng và cung ứng của một doanh nghiệp. Dữ liệu được sinh bằng chương trình, không phải hồ sơ giao dịch của một doanh nghiệp thật. Bộ dữ liệu được tổ chức thành 8 bảng có quan hệ với nhau.

**Bảng là cách tổ chức thông tin; Parquet là định dạng tệp dùng để lưu thông tin đó.** Trong benchmark này, thư mục `lineitem` chứa dữ liệu bảng `lineitem`, được lưu trong `part-0.parquet`; các bảng khác cũng có tệp cùng tên ở thư mục riêng.

```text
tpch_dataset/
├── lineitem/part-0.parquet  → các dòng hàng trong đơn đặt hàng
├── orders/part-0.parquet    → thông tin chung của đơn đặt hàng
├── part/part-0.parquet      → danh mục mặt hàng
└── ...                    → các bảng còn lại
```

`part-0.parquet` là tên tệp phần dữ liệu; chữ `part` trong tên tệp không có nghĩa mọi tệp đó đều chứa bảng `part`. Bảng `part` chỉ là bảng danh mục nằm ở thư mục `part/`.

Parquet lưu dữ liệu theo cột trong các nhóm hàng và kèm metadata mô tả tệp. Ví dụ, từ tệp `lineitem/part-0.parquet`, bộ máy có thể chỉ yêu cầu các cột liên quan đến Q6 thay vì toàn bộ 16 cột. Việc bỏ qua nhóm hàng còn phụ thuộc điều kiện và thống kê; định dạng theo cột không tự bảo đảm truy vấn luôn nhanh.

| Bảng | Một dòng đại diện cho gì? | Thông tin và liên hệ chính |
|---|---|---|
| `region` | Một khu vực địa lý. | Mã và tên khu vực; một khu vực có nhiều quốc gia. |
| `nation` | Một quốc gia. | Thuộc một khu vực qua `n_regionkey`; khách hàng và nhà cung cấp tham chiếu quốc gia. |
| `customer` | Một khách hàng. | Tên, địa chỉ, quốc gia và thông tin khách hàng; một khách hàng có thể đặt nhiều đơn. |
| `orders` | Một đơn đặt hàng. | Khách hàng, ngày đặt, trạng thái và tổng giá trị đơn; liên kết khách hàng bằng `o_custkey`. |
| `lineitem` | Một dòng hàng trong một đơn đặt hàng. | Mã đơn, mã mặt hàng, nhà cung cấp, số lượng, giá trị cả dòng, giảm giá và các ngày giao/nhận. |
| `part` | Một mặt hàng trong danh mục. | Tên, nhãn hiệu, loại, kích thước, bao bì và giá bán lẻ tham chiếu; không phải từng đơn vị hàng vật lý. |
| `supplier` | Một nhà cung cấp. | Mã, tên, địa chỉ, quốc gia và thông tin nhà cung cấp. |
| `partsupp` | Một cặp mặt hàng – nhà cung cấp. | Nhà cung cấp nào cung ứng mặt hàng nào, số lượng sẵn có và giá cung ứng; khóa gồm `ps_partkey`, `ps_suppkey`. |

**Đọc mối liên hệ:** một khách hàng có nhiều đơn; một đơn có nhiều dòng `lineitem`; mỗi dòng tham chiếu một mặt hàng `part` và một nhà cung cấp. Cặp `(l_partkey, l_suppkey)` liên kết với cặp mặt hàng – nhà cung cấp trong `partsupp`.

*Nguồn: TPC Benchmark H, phiên bản 3.0.1, mục 1.2–1.4 về mô hình, lược đồ và định nghĩa bảng; [đặc tả TPC-H](https://www.tpc.org/tpc_documents_current_versions/pdf/tpc-h_v3.0.1.pdf). Định dạng tệp: [Apache Parquet — File Format](https://parquet.apache.org/docs/file-format/).*


### Quy mô dữ liệu TPC-H đang sử dụng

Bảng dưới lấy thông tin trực tiếp từ **tám tệp `part-0.parquet` mà các tệp đăng ký bảng của benchmark sử dụng**. Số dòng là tổng số bản ghi ghi trong metadata Parquet, không phải số dòng ước lượng trong kế hoạch truy vấn.

`pyarrow.parquet.read_metadata(path)` đọc metadata của tệp; `num_rows`, `num_columns` và `num_row_groups` lần lượt cho biết số dòng, số cột và số nhóm hàng. Không giải mã toàn bộ dữ liệu hoặc chạy `COUNT(*)`. Dung lượng lấy bằng `path.stat().st_size` là kích thước tệp Parquet hiện tại; không đồng nhất với quy mô danh nghĩa SF=10 hay bộ nhớ khi thực thi.

*Nguồn API: [PyArrow 14 — read_metadata](https://arrow.apache.org/docs/14.0/python/generated/pyarrow.parquet.read_metadata.html), [FileMetaData](https://arrow.apache.org/docs/14.0/python/generated/pyarrow.parquet.FileMetaData.html).*


In [ ]:
# Đọc thông tin các tệp đang dùng; chưa chạy Q6 hoặc Q17.
import pyarrow.parquet as pq
# Escape dữ liệu trước khi đưa vào HTML notebook.
from html import escape
# Nạp tiện ích phục vụ phần minh họa, chưa chạy truy vấn khi import.
from IPython.display import Code, HTML, Image, display
# Kiểm tra trường hợp thiếu dữ liệu hoặc chưa có giá trị trước khi dùng tiếp.
if str(tpch_dir) not in sys.path:
    # Thực hiện bước chuẩn bị hoặc trình bày với các giá trị đã xác định.
    sys.path.insert(0, str(tpch_dir))
# Dùng chung màu chữ và nền của bảng notebook.
from notebook_theme import REPORT_CSS

# Ý nghĩa từng bảng trong bộ dữ liệu TPC-H.
tpch_table_labels = {
    "lineitem": "Các dòng hàng trong đơn đặt hàng",
    "orders": "Đơn đặt hàng",
    "partsupp": "Quan hệ mặt hàng – nhà cung cấp",
    "part": "Danh mục mặt hàng",
    "customer": "Khách hàng",
    "supplier": "Nhà cung cấp",
    "nation": "Quốc gia",
    "region": "Khu vực",
}

# Định dạng số dòng theo cách nhóm hàng nghìn dễ đọc.
def format_row_count(value):
    # Trả giá trị vừa xác định cho hàm gọi.
    return f"{value:,}".replace(",", ".")

# Chọn đơn vị dung lượng thập phân, không đổi số byte gốc.
def format_file_size(value):
    # Đơn vị thập phân: 1 GB = 1.000.000.000 byte.
    for divisor, unit in ((10**9, "GB"), (10**6, "MB"), (10**3, "kB")):
        # Chọn nhánh phù hợp với dữ liệu hoặc trạng thái vừa đọc.
        if value >= divisor:
            # Trả giá trị vừa xác định cho hàm gọi.
            return f"{value / divisor:.2f}".replace(".", ",") + " " + unit
    # Trả nội dung hiển thị, không chạy thêm SELECT trong bước ghép kết quả.
    return f"{value} byte"

# Lưu metadata của tám tệp thực sự được notebook sử dụng.
tpch_metadata_rows = []
# Duyệt từng mục theo thứ tự để thu dữ liệu hoặc ghép phần trình bày tương ứng.
for table_name in tables:
    # Giá trị trung gian `parquet_path` được dùng ngay ở bước bên dưới.
    parquet_path = data_dir / table_name / "part-0.parquet"
    # Lưu số dòng và nhóm hàng lấy từ footer Parquet, không chạy COUNT.
    metadata = pq.read_metadata(parquet_path)
    # Thêm mục hiện tại vào tập dữ liệu theo đúng thứ tự.
    tpch_metadata_rows.append({
        "table": table_name,
        "rows": metadata.num_rows,
        "columns": metadata.num_columns,
        "row_groups": metadata.num_row_groups,
        "file_bytes": parquet_path.stat().st_size,
        "path": str(parquet_path),
    })

# Sắp xếp theo quy mô; mọi con số bên dưới lấy từ tệp vừa đọc.
tpch_metadata_rows.sort(key=lambda item: item["rows"], reverse=True)
# Giá trị trung gian `lineitem_metadata` được dùng ngay ở bước bên dưới.
lineitem_metadata = next(item for item in tpch_metadata_rows if item["table"] == "lineitem")
# Giá trị trung gian `tpch_metadata_table` được dùng ngay ở bước bên dưới.
tpch_metadata_table = ""
# Giá trị trung gian `tpch_metadata_files` được dùng ngay ở bước bên dưới.
tpch_metadata_files = ""
# Duyệt từng mục theo thứ tự để thu dữ liệu hoặc ghép phần trình bày tương ứng.
for item in tpch_metadata_rows:
    # Tên tệp font macOS theo kiểu chữ cần vẽ.
    name = item["table"]
    # Giá trị trung gian `emphasis` được dùng ngay ở bước bên dưới.
    emphasis = ' class="largest"' if name == "lineitem" else ""
    # Nối thêm nội dung hiện tại, giữ thứ tự các mục đã tạo trước đó.
    tpch_metadata_table += (
        f"<tr{emphasis}><th scope='row'><code>{escape(name)}</code></th>"
        f"<td>{escape(tpch_table_labels[name])}</td>"
        f"<td class='number'>{format_row_count(item['rows'])}</td>"
        f"<td class='number'>{item['columns']}</td>"
        f"<td class='number'>{item['row_groups']}</td>"
        f"<td class='number'>{format_file_size(item['file_bytes'])}</td></tr>"
    )
    # Nối thêm nội dung hiện tại, giữ thứ tự các mục đã tạo trước đó.
    tpch_metadata_files += f"<li><code>{escape(item['path'])}</code></li>"

# Hiển thị dữ liệu hoặc ảnh đã chuẩn bị trong output notebook.
display(HTML(f"""
<section class="tpch-data-scale" aria-label="Quy mô dữ liệu Parquet TPC-H">
<style>
.tpch-data-scale {{font:14px/1.6 system-ui,sans-serif;color:var(--vscode-editor-foreground,#20252b)}}
.tpch-data-scale .lead {{border-left:4px solid #39749b;padding:8px 16px;margin:12px 0 20px}}
.tpch-data-scale .count {{font-size:32px;font-weight:700;line-height:1.3;font-variant-numeric:tabular-nums}}
.tpch-data-scale .table-wrap {{overflow-x:auto}}
.tpch-data-scale table {{width:100%;border-collapse:collapse}}
.tpch-data-scale th,.tpch-data-scale td {{padding:10px 12px;border-bottom:1px solid #8885;text-align:left}}
.tpch-data-scale .number {{text-align:right;white-space:nowrap;font-variant-numeric:tabular-nums}}
.tpch-data-scale thead th {{font-weight:600}}
.tpch-data-scale .largest {{background:#39749b18;font-weight:600}}
.tpch-data-scale summary {{cursor:pointer;font-weight:600}}
.tpch-data-scale li {{overflow-wrap:anywhere}}
</style>
<div class="lead"><div><strong>Bảng lineitem</strong></div>
<div class="count">{format_row_count(lineitem_metadata['rows'])} dòng</div>
<div>{lineitem_metadata['columns']} cột · {lineitem_metadata['row_groups']} nhóm hàng ·
{format_file_size(lineitem_metadata['file_bytes'])} Parquet</div></div>
<div class="table-wrap"><table>
<caption style="text-align:left;padding-bottom:8px">Quy mô thực tế của 8 bảng — số dòng từ metadata tệp</caption>
<thead><tr><th scope="col">Bảng</th><th scope="col">Nội dung</th><th scope="col" class="number">Số dòng</th>
<th scope="col" class="number">Số cột</th><th scope="col" class="number">Nhóm hàng</th>
<th scope="col" class="number">Dung lượng tệp</th></tr></thead>
<tbody>{tpch_metadata_table}</tbody></table></div>
<p>Tổng dung lượng 8 tệp: <strong>{format_file_size(sum(item['file_bytes'] for item in tpch_metadata_rows))}</strong>.
GB/MB/kB là đơn vị thập phân. Nhóm hàng là đơn vị tổ chức dữ liệu trong Parquet, khác phân vùng thực thi của DataFusion.</p>
<details><summary>Đối chiếu đường dẫn 8 tệp đã đọc metadata</summary><ul>{tpch_metadata_files}</ul></details>
</section>
{REPORT_CSS}
"""))


### Một dòng `lineitem` và một dòng `part` khác nhau thế nào?

Ví dụ minh họa: đơn hàng số 100 có dòng 1 mua 17 đơn vị mặt hàng A, dòng 2 mua 3 đơn vị mặt hàng B. Bảng `orders` có một dòng cho đơn 100; bảng `lineitem` có hai dòng hàng. Danh mục `part` mô tả A và B, và có thể được dùng lại cho nhiều đơn khác.

**Một dòng `lineitem` không phải một đơn hàng, không phải một khách hàng và không phải một đơn vị sản phẩm.** Số đơn vị của dòng nằm ở `l_quantity`. Cặp `(l_orderkey, l_linenumber)` xác định dòng hàng trong mô hình TPC-H. Vì vậy, gần 60 triệu dòng lineitem không có nghĩa gần 60 triệu mặt hàng khác nhau hay đơn hàng khác nhau.

**`part` là danh mục mặt hàng.** Một dòng `part` đại diện cho một mã mặt hàng; `p_partkey` liên kết với `l_partkey` ở nhiều dòng bán hàng. Trong Q17, điều kiện nhãn hiệu và bao bì được kiểm tra trên `part`, còn số lượng và giá trị dòng hàng lấy từ `lineitem`.

Ô sau lấy một dòng thật từ đầu tệp lineitem, rồi tìm mặt hàng cùng mã trong tệp part. Đọc lineitem theo lô một dòng; tìm part theo các lô nhỏ đến khi gặp mã. Đây là bước đọc dữ liệu mẫu, tách khỏi bước chỉ đọc metadata và khỏi khoảng đo thời gian truy vấn. Thứ tự vật lý của dòng trong tệp không phải thứ tự được SQL bảo đảm nếu thiếu ORDER BY.


In [ ]:
# Đọc một dòng thật từ tệp lineitem để giải thích ý nghĩa các trường.
from decimal import Decimal
# Nạp tiện ích phục vụ phần minh họa, chưa chạy truy vấn khi import.
import pyarrow.compute as pc
# Giá trị trung gian `sample_lineitem_path` được dùng ngay ở bước bên dưới.
sample_lineitem_path = data_dir / "lineitem" / "part-0.parquet"
# Mở tài nguyên trong khối có quản lý để đóng hoặc dọn tự động khi kết thúc.
with pq.ParquetFile(sample_lineitem_path) as sample_file:
    # Giá trị trung gian `sample_batch` được dùng ngay ở bước bên dưới.
    sample_batch = next(sample_file.iter_batches(batch_size=1, row_groups=[0], use_threads=False), None)
# Kiểm tra trường hợp thiếu dữ liệu hoặc chưa có giá trị trước khi dùng tiếp.
if sample_batch is None or sample_batch.num_rows == 0:
    # Dừng với thông báo rõ ràng thay vì tiếp tục bằng dữ liệu hoặc cấu hình không hợp lệ.
    raise ValueError("Tệp lineitem không có dòng để minh họa.")
# Lưu một dòng hàng thật lấy từ lô đầu của lineitem.
sample_lineitem = sample_batch.to_pylist()[0]

# Tìm mô tả danh mục cùng mã; bộ nhớ được giới hạn theo từng lô.
sample_part = None
# Giá trị trung gian `sample_part_columns` được dùng ngay ở bước bên dưới.
sample_part_columns = ["p_partkey", "p_name", "p_brand", "p_type", "p_size", "p_container", "p_retailprice"]
# Mở tài nguyên trong khối có quản lý để đóng hoặc dọn tự động khi kết thúc.
with pq.ParquetFile(data_dir / "part" / "part-0.parquet") as sample_file:
    # Duyệt từng mục theo thứ tự để thu dữ liệu hoặc ghép phần trình bày tương ứng.
    for sample_batch in sample_file.iter_batches(batch_size=4096, columns=sample_part_columns, use_threads=False):
        # Lưu các dòng part có khóa khớp mặt hàng của lineitem.
        matching_rows = sample_batch.filter(
            pc.equal(sample_batch.column(0), sample_lineitem["l_partkey"])
        )
        # Chọn nhánh phù hợp với dữ liệu hoặc trạng thái vừa đọc.
        if matching_rows.num_rows:
            # Chọn nhánh phù hợp với dữ liệu hoặc trạng thái vừa đọc.
            if matching_rows.num_rows != 1:
                # Dừng với thông báo rõ ràng thay vì tiếp tục bằng dữ liệu hoặc cấu hình không hợp lệ.
                raise ValueError("Mã mặt hàng xuất hiện nhiều lần trong lô mẫu.")
            # Lưu bản ghi danh mục cùng mã với dòng hàng mẫu.
            sample_part = matching_rows.to_pylist()[0]
            # Đã tìm thấy mục cần dùng, dừng vòng lặp để tránh đọc thêm.
            break
# Kiểm tra trường hợp thiếu dữ liệu hoặc chưa có giá trị trước khi dùng tiếp.
if sample_part is None:
    # Dừng với thông báo rõ ràng thay vì tiếp tục bằng dữ liệu hoặc cấu hình không hợp lệ.
    raise ValueError("Không tìm thấy mặt hàng tương ứng trong part.")

# Định dạng số mẫu theo cách viết thập phân tiếng Việt.
def format_sample_number(value):
    # Trả giá trị vừa xác định cho hàm gọi.
    return format(value, ",").translate(str.maketrans({",": ".", ".": ","}))

# Giá trị trung gian `sample_lineitem_fields` được dùng ngay ở bước bên dưới.
sample_lineitem_fields = {
    "l_orderkey": "Mã đơn đặt hàng; nối với orders.o_orderkey.",
    "l_linenumber": "Số thứ tự dòng hàng trong đơn; cùng l_orderkey xác định dòng.",
    "l_partkey": "Mã mặt hàng; nối với part.p_partkey.",
    "l_suppkey": "Mã nhà cung cấp của dòng hàng này.",
    "l_quantity": "Số đơn vị hàng trên dòng này.",
    "l_extendedprice": "Giá trị cả dòng trước giảm giá và thuế; đã tính số lượng, không phải đơn giá.",
    "l_discount": "Tỷ lệ giảm giá dạng thập phân; 0,04 tương ứng 4%.",
    "l_tax": "Tỷ lệ thuế dạng thập phân; 0,02 tương ứng 2%.",
    "l_returnflag": "Cờ trạng thái trả hàng theo mã của TPC-H.",
    "l_linestatus": "Cờ trạng thái dòng hàng theo mã của TPC-H.",
    "l_shipdate": "Ngày hàng được giao đi.",
    "l_commitdate": "Ngày cam kết giao hàng.",
    "l_receiptdate": "Ngày nhận hàng.",
    "l_shipinstruct": "Chỉ dẫn giao hàng.",
    "l_shipmode": "Phương thức vận chuyển.",
    "l_comment": "Ghi chú của dòng hàng.",
}
# Giá trị trung gian `sample_part_fields` được dùng ngay ở bước bên dưới.
sample_part_fields = {
    "p_partkey": "Mã mặt hàng trong danh mục, bằng l_partkey của dòng mẫu.",
    "p_name": "Tên mặt hàng được sinh trong TPC-H.",
    "p_brand": "Nhãn hiệu; đây là cột được lọc trong Q17.",
    "p_type": "Loại mặt hàng.",
    "p_size": "Kích thước quy ước của mặt hàng.",
    "p_container": "Loại bao bì; đây là cột được lọc trong Q17.",
    "p_retailprice": "Giá bán lẻ tham chiếu cho một đơn vị trong danh mục.",
}

# Ghép giá trị của dòng mẫu với ý nghĩa từng cột.
def sample_table_html(record, meanings):
    # Lưu các dòng kết quả hoặc HTML đang được ghép.
    rows = ""
    # Duyệt từng mục theo thứ tự để thu dữ liệu hoặc ghép phần trình bày tương ứng.
    for name, meaning in meanings.items():
        # Giá trị trung gian `value` được dùng ngay ở bước bên dưới.
        value = record[name]
        # Giá trị trung gian `shown` được dùng ngay ở bước bên dưới.
        shown = format_sample_number(value) if isinstance(value, (int, Decimal)) else str(value)
        # Nối thêm nội dung hiện tại, giữ thứ tự các mục đã tạo trước đó.
        rows += (f"<tr><th scope='row'><code>{escape(name)}</code></th>"
                 f"<td>{escape(shown)}</td><td>{escape(meaning)}</td></tr>")
    # Trả giá trị vừa xác định cho hàm gọi.
    return ("<div class='table-wrap'><table><thead><tr><th>Cột</th><th>Giá trị thật</th>"
            "<th>Ý nghĩa</th></tr></thead><tbody>" + rows + "</tbody></table></div>")

# Giá trị trung gian `sample_quantity` được dùng ngay ở bước bên dưới.
sample_quantity = sample_lineitem["l_quantity"]
# Giá trị trung gian `sample_value` được dùng ngay ở bước bên dưới.
sample_value = sample_lineitem["l_extendedprice"]
# Giá trị trung gian `sample_unit_price` được dùng ngay ở bước bên dưới.
sample_unit_price = sample_value / sample_quantity if sample_quantity else None
# Giá trị trung gian `sample_discount_amount` được dùng ngay ở bước bên dưới.
sample_discount_amount = sample_value * sample_lineitem["l_discount"]
# Hiển thị dữ liệu hoặc ảnh đã chuẩn bị trong output notebook.
display(HTML(f"""<section class="tpch-data-scale" aria-label="Ý nghĩa dòng hàng và mặt hàng">
<h4>Dòng hàng thật từ lineitem/part-0.parquet</h4>
<p>Đơn <strong>{sample_lineitem['l_orderkey']}</strong>, dòng số <strong>{sample_lineitem['l_linenumber']}</strong>:
đặt <strong>{format_sample_number(sample_quantity)}</strong> đơn vị của mặt hàng
<strong>{sample_lineitem['l_partkey']}</strong> từ nhà cung cấp <strong>{sample_lineitem['l_suppkey']}</strong>.
Giá trị cả dòng trước giảm giá và thuế là <strong>{format_sample_number(sample_value)}</strong>.</p>
{sample_table_html(sample_lineitem, sample_lineitem_fields)}
<h4>Mặt hàng tương ứng trong part/part-0.parquet</h4>
<p><code>lineitem.l_partkey = part.p_partkey = {sample_part['p_partkey']}</code>.
Mã này liên kết dòng đặt hàng với thông tin danh mục.</p>
{sample_table_html(sample_part, sample_part_fields)}
<p>Đơn giá suy ra từ dòng mẫu: {format_sample_number(sample_unit_price) if sample_unit_price is not None else 'không xác định vì số lượng bằng 0'}.
Tiền giảm giá của dòng: {format_sample_number(sample_value)} × {format_sample_number(sample_lineitem['l_discount'])}
= <strong>{format_sample_number(sample_discount_amount)}</strong>.
Đây là dữ liệu nguồn chưa áp điều kiện Q6/Q17. Các giá trị tiền theo bộ TPC-H, không tự quy đổi thành đồng Việt Nam.</p>
<p>Q6 đọc ngày giao đi, số lượng và tỷ lệ giảm để chọn dòng, rồi cộng tiền giảm giá.
Q17 dùng thông tin danh mục part để chọn mặt hàng, sau đó so số lượng từng dòng lineitem với trung bình của chính mặt hàng đó.</p>
</section>{REPORT_CSS}"""))


In [ ]:
# Chuyển đường dẫn Parquet trong chuỗi SQL thành đường dẫn tuyệt đối.
def absolute_paths(sql):
    # Hàm này được gọi cho mỗi đường dẫn khớp mẫu bên dưới.
    def replace(match):
        # group(1) là đường dẫn bên trong dấu nháy đơn; resolve chuẩn hóa đường dẫn.
        path = (tpch_dir / match.group(1)).resolve()
        # SQL dùng hai dấu nháy đơn để biểu diễn một dấu nháy trong tên tệp.
        return "'" + str(path).replace("'", "''") + "'"
    # Chỉ thay chuỗi kết thúc bằng .parquet nằm trong dấu nháy đơn.
    return re.sub(r"'([^']+\.parquet)'", replace, sql)

# Cấu hình một phân vùng thực thi cho DataFusion; không phải giới hạn cứng mọi luồng hệ thống.
config = SessionConfig().with_target_partitions(1)
# Tạo phiên DataFusion trong chương trình Python, dùng cấu hình tài nguyên mặc định.
ctx = SessionContext(config, RuntimeConfig())
# Đọc mỗi dòng đăng ký bảng Parquet từ tệp của tác giả.
for sql in (tpch_dir / "create-single-datafusion.sql").read_text().splitlines():
    # Bỏ dòng trống nếu có.
    if sql.strip():
        # Đăng ký bảng theo đường dẫn tuyệt đối; chưa chạy Q6 hay Q17.
        ctx.sql(absolute_paths(sql))

# Nếu chạy lại ô này thì đóng kết nối DuckDB của lượt trước.
if "con" in globals():
    # Đóng tài nguyên khi không dùng nữa.
    con.close()
# Tạo cơ sở dữ liệu tạm trong RAM; dữ liệu bảng vẫn được đọc từ Parquet.
con = duckdb.connect(":memory:")
# Cấu hình DuckDB sử dụng một luồng xử lý truy vấn.
con.execute("PRAGMA threads=1")
# Không đổi thanh tiến trình: DuckDB 0.9.1 có thể đòi ipywidgets khi gọi lệnh đó trong Jupyter.
# Đọc toàn bộ tệp tạo view DuckDB, chuyển đường dẫn rồi đăng ký các view.
con.execute(absolute_paths((tpch_dir / "create-single-duckdb.sql").read_text()))
# Thông báo hoàn tất bước chuẩn bị, không phải kết quả truy vấn.
print("Đã đăng ký cùng 8 bảng Parquet; DataFusion 1 phân vùng, DuckDB 1 luồng.")


### Đối chiếu kế hoạch và cách lấy các giai đoạn

`compare_plans` là tiện ích trình bày bổ sung, không phải hàm có sẵn của DataFusion hoặc mã benchmark của tác giả.

**Lời gọi DataFusion được sử dụng:**

```python
explain_query = ctx.sql("EXPLAIN VERBOSE " + sql)
explain_batches = explain_query.collect()
rows = [
    row
    for batch in explain_batches
    for row in batch.to_pylist()
]
```

| Thành phần | Ý nghĩa |
|---|---|
| `ctx.sql(...)` | Gửi câu lệnh SQL cho DataFusion. Ở đây câu lệnh là `EXPLAIN VERBOSE` của Q6 hoặc Q17. |
| `EXPLAIN VERBOSE` | Yêu cầu mô tả chi tiết quá trình lập kế hoạch: ban đầu, sau các quy tắc và cuối cùng. Đây là câu lệnh SQL, không phải một hàm tối ưu do tiện ích viết. |
| `collect()` | Thực thi câu lệnh EXPLAIN và thu các lô chứa mô tả kế hoạch; không thu doanh thu của Q6 hoặc Q17. |
| `batch.to_pylist()` | Chuyển một lô Arrow sang danh sách dòng Python để trình bày. |
| `plan_type` và `plan` | Tên trạng thái và nội dung kế hoạch do DataFusion trả về. |

DataFusion Python 32.0.0 cũng có `ctx.sql(sql).explain(verbose=True, analyze=False)` để in mô tả chi tiết. Tiện ích dùng câu SQL EXPLAIN với `collect()` để lấy các dòng mô tả và bố trí hai bộ máy cạnh nhau, thay vì chỉ in bảng văn bản.

Ví dụ, `logical_plan after type_coercion` là trạng thái sau bước điều chỉnh kiểu, không phải sau khi xử lý một phần dữ liệu bán hàng. `SAME TEXT AS ABOVE` nghĩa là nội dung kế hoạch không đổi so với trạng thái trước đó. Một quy tắc có thể xuất hiện nhiều lần vì bộ tối ưu chạy nhiều lượt. Tiện ích giữ thứ tự và tên do DataFusion trả về.

Các bước **logic** ứng với số **3–4** trên hình; `initial_physical_plan`, các quy tắc **vật lý** và `physical_plan` ứng với **5–6**. Chỉ khi thực thi Q6 hoặc Q17 thật mới có kết quả và thời gian chạy truy vấn ở **7–8**. EXPLAIN thông thường không cung cấp thời gian, số dòng thực tế hoặc bộ nhớ của từng toán tử.

**DuckDB:** `con.execute("EXPLAIN " + sql).fetchall()`, với `explain_output='all'`, cung cấp `logical_plan`, `logical_opt` và `physical_plan`. Tiện ích khôi phục thiết lập này sau khi đọc; không tự dựng các giai đoạn tối ưu DuckDB chưa được API trả về.

Thứ tự hiển thị: **kế hoạch ban đầu → logic sau tối ưu → vật lý cuối → đối chiếu toán tử → chi tiết các giai đoạn → đầu ra nguyên bản**. Mỗi mục có chú giải tiếng Việt; tên kỹ thuật được giữ để đối chiếu. `EC` của DuckDB là số dòng ước lượng.

Các bảng giải thích Q6 và Q17 bên dưới có số liệu từ một lượt chẩn đoán riêng, được lưu cùng ngày và kèm đường dẫn nguồn. `compare_plans` vẫn chỉ lấy EXPLAIN, không tự chạy lại truy vấn để lấy số liệu này.

*Nguồn: DataFusion Python [32.0.0, SessionContext và DataFrame](https://github.com/apache/datafusion-python/tree/32.0.0/src); cú pháp [EXPLAIN của DataFusion 32.0.0](https://github.com/apache/datafusion/blob/32.0.0/docs/source/user-guide/sql/explain.md); bài báo, mục 5.4, 6.1, tr. 9–10.*

**Định vị trên hình:** mục 0 trong bảng mở sẵn Hình 2 có số khoanh. Mỗi mục 1–4 có hình đánh dấu vùng tương ứng ngay trước kế hoạch; mục 5 gắn nhãn vị trí cho từng trạng thái thật. Các số 1–8 giữ như hình đầu notebook; số 9 đánh dấu điểm mở rộng `ExecutionPlan Rewrite` dùng ở phần quy tắc Rust Q17. Chú giải này ánh xạ DataFusion, không phải sơ đồ nội bộ của DuckDB.


In [ ]:
# Nạp các tiện ích trình bày từ thư mục nằm cạnh notebook.
from plan_comparison import compare_plans
# Thu hồ sơ DuckDB và so kết quả Q6, không chạy SELECT khi import.
from q6_walkthrough import profile_duckdb_q6, q6_diagnostics, q6_control_result
# Gọi chương trình Rust và trình bày đối chứng Q17 ở các mục phía sau.
from q17_extension import run_q17_rule, show_rule_comparison, show_estimation_decisions


## 1. Q6 — lọc và tổng hợp một bảng


In [ ]:
# Hiển thị SQL có xuống dòng; câu thực thi vẫn lấy từ queries.sql.
q6_display = """
select
    sum(l_extendedprice * l_discount) as revenue
from lineitem
where l_shipdate >= date '1994-01-01'
  and l_shipdate < date '1995-01-01'
  and l_discount between 0.06 - 0.01 and 0.06 + 0.01
  and l_quantity < 24;
"""
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print(q6_display)


**Ý nghĩa truy vấn**

Q6 tính tổng tiền giảm giá của các dòng hàng thỏa điều kiện. Bảng `lineitem` chứa các dòng hàng; `l_extendedprice` là giá trị cả dòng trước giảm giá, đã tính số lượng; `l_discount` là tỷ lệ giảm giá.

Các điều kiện giữ lại dòng có ngày giao trong năm 1994, tỷ lệ giảm giá từ 0,05 đến 0,07, bao gồm hai đầu mút, và số lượng nhỏ hơn 24. Tất cả điều kiện phải đồng thời đúng.

Biểu thức `l_extendedprice * l_discount` tính tiền giảm giá của từng dòng. `SUM` cộng các giá trị này thành một kết quả mang tên `revenue`. Ví dụ, dòng có giá trị 100 và tỷ lệ giảm 0,06 đóng góp 6 vào tổng. Truy vấn không có `GROUP BY`; nếu không có giá trị để cộng, `SUM` trả `NULL`.

**Ý nghĩa trong hoạt động bán hàng**

Một doanh nghiệp muốn xem xét chính sách giảm giá và đặt câu hỏi: **Nếu không áp dụng mức giảm từ 5% đến 7% cho các dòng hàng có số lượng dưới 24, đã giao trong năm 1994, thì doanh thu có thể tăng thêm bao nhiêu?** Q6 cộng số tiền giảm giá của đúng những dòng hàng này để lượng hóa kịch bản đó. Đây là mục đích nghiệp vụ được mô tả trong đặc tả TPC-H, với tên gọi dự báo thay đổi doanh thu.

Ví dụ minh họa bằng đồng, giả sử cả ba dòng hàng đều có ngày giao trong năm 1994:

| Dòng hàng | Số lượng | Giá trị cả dòng trước giảm | Tỷ lệ giảm | Tiền được cộng vào kết quả Q6 |
|---|---:|---:|---:|---:|
| A | 10 | 100.000 đồng | 6% | 6.000 đồng |
| B | 20 | 200.000 đồng | 5% | 10.000 đồng |
| C | 30 | 300.000 đồng | 6% | Không tính vì số lượng không dưới 24 |

Kết quả là **16.000 đồng**. Đây là tổng tiền giảm giá của các dòng đủ điều kiện, đồng thời là phần doanh thu có thể thu thêm trong giả định bỏ các khoản giảm giá đó mà giá trị trước giảm và lượng hàng bán không đổi. Con số này giúp doanh nghiệp xem xét quy mô khoản giảm giá trước khi quyết định điều chỉnh chính sách.

Tên cột `revenue` không có nghĩa kết quả là toàn bộ doanh thu bán hàng hoặc lợi nhuận. Truy vấn không tính giá vốn, không dự báo phản ứng của khách hàng và không cập nhật chính sách giảm giá. Điều kiện số lượng áp dụng cho từng dòng hàng, không phải tổng số lượng của cả đơn hàng. Ví dụ bằng đồng chỉ dùng để minh họa, không xác định đơn vị tiền của dữ liệu TPC-H.

*Nguồn: Transaction Processing Performance Council, TPC Benchmark H, phiên bản 3.0.1, mục 2.4.6–2.4.6.1, tr. 38. [Đặc tả TPC-H](https://www.tpc.org/tpc_documents_current_versions/pdf/tpc-h_v3.0.1.pdf).*


### 1.1. Q6 trong hình kiến trúc

**Luồng lập kế hoạch:** SQL (1) và thông tin bảng Parquet (2) → LogicalPlan (3) → tối ưu logic (4) → ExecutionPlan và tối ưu vật lý (5–6).

**Luồng chạy dữ liệu:** khi gọi `collect()`, các toán tử trong ExecutionPlan đọc nguồn (2), đánh giá điều kiện và tổng hợp (7), truyền các lô Arrow và thu kết quả (8). Hình kiến trúc mô tả các thành phần; các mũi tên lập kế hoạch không phải đường di chuyển của từng dòng dữ liệu.

Q6 dùng bốn cột `l_quantity`, `l_extendedprice`, `l_discount`, `l_shipdate`. Parquet là định dạng trên đĩa; Arrow là biểu diễn các lô trong bộ nhớ DataFusion. DuckDB có đường đọc Parquet và xử lý vector riêng; không lấy việc DataFusion dùng Arrow làm bằng chứng tự động nhanh hơn.

*Nguồn: [Lamb và cộng sự, Hình 2, mục 5](https://doi.org/10.1145/3626246.3653368). Số khoanh tròn và phần chú giải do nhóm bổ sung.*


In [ ]:
# Hiển thị ảnh có sẵn, không chạy truy vấn.
display(Image(filename=str(tpch_dir / "q6-assets" / "01-q6-kien-truc.png"), width=1100))


### 1.2. Xác nhận cấu hình trước khi chạy

`filter pushdown` nghĩa là chuyển điều kiện lọc xuống gần nguồn dữ liệu. Cần phân biệt **bỏ nhóm hàng bằng thống kê min/max** với **lọc các dòng trong lúc giải mã**.

DataFusion 32 dùng `datafusion.execution.parquet.pushdown_filters` để điều khiển lọc trong giải mã; mặc định `false`. `datafusion.execution.parquet.pruning` là tùy chọn khác, mặc định `true`.

DuckDB 0.9.1 cho bộ đọc Parquet nhận điều kiện lọc. Danh sách `disabled_optimizers` rỗng nghĩa là chưa tắt quy tắc nào; phần này sẽ đặt lại giá trị rỗng để thu lượt Q6 mặc định.


In [ ]:
# Bật bảng thông tin cấu hình để DataFusion 32 hỗ trợ SHOW; không đổi cách lọc.
ctx.sql("SET datafusion.catalog.information_schema = true").collect()
# SHOW đọc cấu hình của chính phiên DataFusion đang dùng, không chạy Q6.

# Giá trị trung gian `q6_df_filter_setting` được dùng ngay ở bước bên dưới.
q6_df_filter_setting = ctx.sql("SHOW datafusion.execution.parquet.pushdown_filters").collect()[0].to_pylist()[0]["value"]
# Giá trị trung gian `q6_df_pruning_setting` được dùng ngay ở bước bên dưới.
q6_df_pruning_setting = ctx.sql("SHOW datafusion.execution.parquet.pruning").collect()[0].to_pylist()[0]["value"]
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print("DataFusion — pushdown_filters:", q6_df_filter_setting)
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print("DataFusion — pruning:", q6_df_pruning_setting)

# Chọn nhánh phù hợp với dữ liệu hoặc trạng thái vừa đọc.
if q6_df_filter_setting.lower() != "false":
    # Dừng với thông báo rõ ràng thay vì tiếp tục bằng dữ liệu hoặc cấu hình không hợp lệ.
    raise RuntimeError("Phiên hiện tại đã bật lọc trong giải mã. Chạy lại ô tạo ctx để bắt đầu bằng cấu hình mặc định.")

# Đặt DuckDB về mặc định trước khi lấy số đo gốc.
con.execute("SET disabled_optimizers=''")
# Lưu kết quả đối chứng DuckDB, để None nếu chưa chạy.
q6_duck_changed = None
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print("DuckDB — disabled_optimizers:", repr(con.execute("SELECT current_setting('disabled_optimizers')").fetchone()[0]))


### Thực thi Q6 bằng DataFusion

`ctx.sql(...)` tạo đối tượng truy vấn; `collect()` thực thi và thu kết quả. Thời gian được đo bao quanh hai lời gọi này.


In [ ]:
# Lưu mốc bắt đầu trước khi gửi SQL.
start = perf_counter()
# DataFusion lập kế hoạch và thực thi Q6; collect thu các RecordBatch kết quả.
q6_df_batches = ctx.sql(q6_sql).collect()
# Lấy mốc kết thúc trừ mốc bắt đầu; đơn vị giây.
q6_df_seconds = perf_counter() - start
# Lấy lô đầu, cột đầu, giá trị đầu; as_py chuyển giá trị Arrow sang Python.
q6_df_value = q6_df_batches[0].column(0)[0].as_py()
# In kết quả đã thu, không chạy SQL lần nữa.
print("DataFusion — Q6:", q6_df_value)
# Hiển thị thời gian với ba chữ số sau dấu thập phân.
print(f"Thời gian một lượt: {q6_df_seconds:.3f} giây")


**Kế hoạch DataFusion Q6**

`ParquetExec` cung cấp các cột cần thiết; `FilterExec` giữ các dòng thỏa điều kiện. `CoalesceBatchesExec` gom các lô nhỏ trước các bước tiếp theo. `AggregateExec` tính tổng biểu thức nhân; `ProjectionExec` chọn các cột hoặc đặt tên đầu ra.

### Thực thi Q6 bằng DuckDB

`con.execute(...).fetchall()` thực thi SQL và thu toàn bộ kết quả. Các khung nhìn DuckDB tham chiếu cùng các tệp Parquet được DataFusion sử dụng.


In [ ]:
# Tiện ích chỉ chạy SELECT Q6 một lần, bật JSON profiling để lấy thời gian toán tử.
# EXPLAIN lấy kế hoạch trước; không nằm trong khoảng đo execute + fetchall.
q6_duck_default = profile_duckdb_q6(con, q6_sql)
# Giá trị trung gian `q6_duck_rows` được dùng ngay ở bước bên dưới.
q6_duck_rows = q6_duck_default["rows"]
# Giá trị trung gian `q6_duck_seconds` được dùng ngay ở bước bên dưới.
q6_duck_seconds = q6_duck_default["seconds"]
# Giá trị trung gian `q6_duck_value` được dùng ngay ở bước bên dưới.
q6_duck_value = q6_duck_rows[0][0]
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print("DuckDB — Q6:", q6_duck_value)
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print(f"Thời gian một lượt: {q6_duck_seconds:.3f} giây")
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print("Hai kết quả bằng nhau:", q6_df_value == q6_duck_value)


### 1.3. Xem kế hoạch hai bộ máy

Mở mục **3. Kế hoạch vật lý cuối** trong bảng. DataFusion có `FilterExec` riêng; DuckDB mặc định đưa điều kiện vào `READ_PARQUET`. Đây là khác biệt cấu trúc, chưa đủ kết luận hiệu năng. Phần tiếp theo thu số đo để xác định công việc tốn thời gian.


In [ ]:
# EXPLAIN hai bộ máy và hiển thị trong một bảng; không chạy lại Q6 để lấy kết quả.
# Bấm mở các mục bên dưới bảng để xem kế hoạch đầy đủ và các bước tối ưu.
display(compare_plans(ctx, con, q6_sql, "Q6", data_dir=data_dir))

# ctx.sql("EXPLAIN " + q6_sql).collect()


### 1.4. Phát hiện công việc tốn thời gian

Ô dưới chạy **thêm một lần Q6 trên DataFusion** bằng `EXPLAIN ANALYZE VERBOSE`, thu kế hoạch kèm bộ đếm. `collect()` ở đây thu các dòng mô tả kế hoạch và số liệu, không phải giá trị doanh thu. Hồ sơ DuckDB đã được thu ở lần SELECT phía trên nên không chạy DuckDB lại.

Đọc `READ_PARQUET.timing` trước, rồi so với `PROJECTION` và `UNGROUPED_AGGREGATE` **trong cùng hồ sơ DuckDB**. Nếu phần đọc/lọc lớn vượt trội, đó là vùng cần khảo sát. Với DataFusion, xem `ParquetExec.time_elapsed_processing`, `FilterExec.elapsed_compute` và `AggregateExec.elapsed_compute`; các bộ đếm có phạm vi khác nhau, không cộng chúng để tính lại thời gian tổng.

`row_groups_pruned=0` nghĩa là không bỏ được nhóm hàng; không có nghĩa bộ lọc không loại dòng. `pushdown_rows_filtered=0` phải đọc cùng cấu hình bộ đọc và số dòng của `FilterExec`.


In [ ]:
# ANALYZE thực thi Q6 thêm một lần để đo, khác với EXPLAIN chỉ lập kế hoạch.
q6_df_analyze_batches = ctx.sql("EXPLAIN ANALYZE VERBOSE " + q6_sql).collect()
# Giá trị trung gian `q6_df_analyze_rows` được dùng ngay ở bước bên dưới.
q6_df_analyze_rows = [row for batch in q6_df_analyze_batches for row in batch.to_pylist()]
# Bảng và thanh thời gian lấy số liệu vừa thu, không dùng thời gian điền sẵn.
display(q6_diagnostics(q6_df_analyze_rows, q6_duck_default))


### 1.5. Đối chiếu đường thực thi với mã nguồn đúng phiên bản

Hồ sơ toán tử khoanh vùng công việc tốn thời gian; mã nguồn giải thích công việc đó được thực hiện ra sao. Ảnh phía dưới lấy từ **DataFusion 32.0.0** và **DuckDB 0.9.1**, có số dòng và liên kết cố định theo commit. Đây là ảnh nguyên văn mã, không phải mã minh họa viết lại.

**DataFusion — cấu hình lọc trong giải mã:** `pushdown_filters=false` làm bộ đọc không gắn `RowFilter` theo tùy chọn này. `FilterExec` bên ngoài vẫn kiểm tra điều kiện Q6; bỏ nhóm hàng bằng thống kê là cơ chế khác.


In [ ]:
# Hiển thị ảnh có sẵn, không chạy truy vấn.
display(Image(filename=str(tpch_dir / "q6-assets" / "02-datafusion-cau-hinh.png"), width=1100))


**Nguồn ảnh mã:** ảnh kết xuất từ nguyên văn tệp đã checkout, giữ số dòng gốc; vùng tô màu do nhóm bổ sung.

1. [`datafusion/common/src/config.rs`, dòng 284–286](https://github.com/apache/datafusion/blob/eca48dae2447a67fcf30313c956e6c39cf739d48/datafusion/common/src/config.rs#L284-L286) — commit `eca48dae2447`.
2. [`datafusion/core/src/datasource/physical_plan/parquet.rs`, dòng 211–215](https://github.com/apache/datafusion/blob/eca48dae2447a67fcf30313c956e6c39cf739d48/datafusion/core/src/datasource/physical_plan/parquet.rs#L211-L215) — commit `eca48dae2447`.
3. [`datafusion/core/src/datasource/physical_plan/parquet.rs`, dòng 462–475](https://github.com/apache/datafusion/blob/eca48dae2447a67fcf30313c956e6c39cf739d48/datafusion/core/src/datasource/physical_plan/parquet.rs#L462-L475) — commit `eca48dae2447`.


**DataFusion — bộ lọc riêng:** `predicate.evaluate(batch)` đánh giá điều kiện trên lô Arrow; `into_array` tạo biểu diễn mảng; `filter_record_batch` chọn các dòng thỏa điều kiện. Công việc này nằm trong phần biểu thức/toán tử thực thi (7) của hình kiến trúc.


In [ ]:
# Hiển thị ảnh có sẵn, không chạy truy vấn.
display(Image(filename=str(tpch_dir / "q6-assets" / "03-datafusion-filterexec.png"), width=1100))


**Nguồn ảnh mã:** ảnh kết xuất từ nguyên văn tệp đã checkout, giữ số dòng gốc; vùng tô màu do nhóm bổ sung.

1. [`datafusion/physical-plan/src/filter.rs`, dòng 303–315](https://github.com/apache/datafusion/blob/eca48dae2447a67fcf30313c956e6c39cf739d48/datafusion/physical-plan/src/filter.rs#L303-L315) — commit `eca48dae2447`.


**DuckDB — vì sao điều kiện nằm trong READ_PARQUET:** `table_function.filter_pushdown=true` khai báo khả năng của bộ đọc. `FilterPushdown::PushdownGet` chuyển điều kiện phù hợp thành `table_filters` của phép quét. Đây là mã của bộ tối ưu logic; khi thực thi, Parquet reader dùng các điều kiện đã nhận.


In [ ]:
# Hiển thị ảnh có sẵn, không chạy truy vấn.
display(Image(filename=str(tpch_dir / "q6-assets" / "04-duckdb-cau-hinh.png"), width=1100))


**Nguồn ảnh mã:** ảnh kết xuất từ nguyên văn tệp đã checkout, giữ số dòng gốc; vùng tô màu do nhóm bổ sung.

1. [`extension/parquet/parquet_extension.cpp`, dòng 175–180](https://github.com/duckdb/duckdb/blob/401c8061c6ece35949cac58c7770cc755710ca86/extension/parquet/parquet_extension.cpp#L175-L180) — commit `401c8061c6ec`.
2. [`src/optimizer/pushdown/pushdown_get.cpp`, dòng 52–56](https://github.com/duckdb/duckdb/blob/401c8061c6ece35949cac58c7770cc755710ca86/src/optimizer/pushdown/pushdown_get.cpp#L52-L56) — commit `401c8061c6ec`.


**DuckDB — phần đọc/lọc được khoanh vùng:** `child_reader->Read(...)` đọc và giải mã cột phục vụ điều kiện; `ApplyFilter(...)` cập nhật các dòng được giữ. Trong `TemplatedFilterOperation`, `v_ptr[i]` là giá trị tại vị trí `i` của vector; `constant` là hằng số điều kiện; `OP::Operation` là phép so sánh; `filter_mask[i]` ghi dòng còn được giữ hay không. `&&` có thể bỏ phép so sánh khi dòng đã bị loại, nhưng vòng lặp vẫn đi qua các vị trí.

Ảnh giải thích mã thuộc đường đọc và lọc. Dùng hồ sơ thực thi vừa thu và lượt đối chứng phía dưới để đánh giá trên máy đang chạy. Không quy toàn bộ chênh lệch thời gian cho riêng dòng cập nhật mặt nạ. DuckDB vẫn xử lý theo vector.


In [ ]:
# Hiển thị ảnh có sẵn, không chạy truy vấn.
display(Image(filename=str(tpch_dir / "q6-assets" / "05-duckdb-doc-va-loc.png"), width=1100))


**Nguồn ảnh mã:** ảnh kết xuất từ nguyên văn tệp đã checkout, giữ số dòng gốc; vùng tô màu do nhóm bổ sung.

1. [`extension/parquet/parquet_reader.cpp`, dòng 945–955](https://github.com/duckdb/duckdb/blob/401c8061c6ece35949cac58c7770cc755710ca86/extension/parquet/parquet_reader.cpp#L945-L955) — commit `401c8061c6ec`.
2. [`extension/parquet/parquet_reader.cpp`, dòng 688–703](https://github.com/duckdb/duckdb/blob/401c8061c6ece35949cac58c7770cc755710ca86/extension/parquet/parquet_reader.cpp#L688-L703) — commit `401c8061c6ec`.


**DuckDB — đường FILTER riêng dùng để đối chứng:** `SelectExpression` đánh giá điều kiện và chọn vị trí hợp lệ. Nếu giữ mọi dòng, `Reference` tham chiếu đầu vào; nếu chỉ giữ một phần, `Slice` lấy các vị trí đó. Tắt quy tắc `filter_pushdown` không bỏ điều kiện WHERE; nó chuyển vị trí và cách thực hiện việc lọc.


In [ ]:
# Hiển thị ảnh có sẵn, không chạy truy vấn.
display(Image(filename=str(tpch_dir / "q6-assets" / "06-duckdb-filter-rieng.png"), width=1100))


**Nguồn ảnh mã:** ảnh kết xuất từ nguyên văn tệp đã checkout, giữ số dòng gốc; vùng tô màu do nhóm bổ sung.

1. [`src/execution/operator/filter/physical_filter.cpp`, dòng 42–53](https://github.com/duckdb/duckdb/blob/401c8061c6ece35949cac58c7770cc755710ca86/src/execution/operator/filter/physical_filter.cpp#L42-L53) — commit `401c8061c6ec`.


### 1.6. Hình luồng trước và sau khi đổi cấu hình

Hàng DataFusion đối chiếu vùng nguồn (2), biểu thức/toán tử (7), lô kết quả (8) của Hình 2. Hai hàng DuckDB thể hiện chức năng tương ứng; đây không phải sơ đồ kiến trúc DuckDB của bài báo. Độ dài các ô không biểu thị thời gian.

Q6 dùng điều kiện ngày giao, mức giảm giá và số lượng. Việc lọc trong bộ đọc có thể tiết kiệm giải mã các giá trị không cần thiết, đồng thời phát sinh chi phí kiểm tra và điều khiển lựa chọn dòng trong đường đọc. Phép đối chứng phía dưới kiểm tra **tổng hiệu quả của hai đường thực thi trên chính dữ liệu này**, không đo riêng một vòng lặp C++.

Đường đọc và lọc riêng có thể thuận lợi trong thí nghiệm Q6 đã thu. Không kết luận lọc sớm luôn chậm hoặc không loại được dòng; kết quả đo được là căn cứ đánh giá hiệu quả của thay đổi.


In [ ]:
# Hiển thị ảnh có sẵn, không chạy truy vấn.
display(Image(filename=str(tpch_dir / "q6-assets" / "07-q6-luong-truoc-sau.png"), width=1100))


### 1.7. Thay đổi cấu hình lọc của DuckDB

**Chỉ sửa dòng lệnh trong ô ngay dưới**, từ:

```python
con.execute("SET disabled_optimizers=''")
```

thành:

```python
con.execute("SET disabled_optimizers='filter_pushdown'")
```

Sau đó chạy ô cấu hình rồi ô đối chứng tiếp theo. `disabled_optimizers` là danh sách quy tắc bị tắt trong phiên; không phải `filter_pushdown=false`, cũng không sửa trực tiếp `table_function.filter_pushdown` trong C++.

DataFusion giữ cấu hình mặc định. Không thay SQL Q6, tệp Parquet hoặc số luồng. Đây là thí nghiệm cấu hình, chưa phải tự cài một quy tắc tối ưu hay viết lại mã bộ máy.


In [ ]:
# Mặc định không tắt quy tắc nào. Đổi chuỗi rỗng thành filter_pushdown để đối chứng.
con.execute("SET disabled_optimizers=''")
# Đọc lại cấu hình của chính kết nối, chưa chạy truy vấn.
print("Quy tắc bị tắt:", repr(con.execute("SELECT current_setting('disabled_optimizers')").fetchone()[0]))


In [ ]:
# Nếu chưa sửa ô cấu hình, bỏ qua để không chạy lại Q6 ngoài ý muốn.
q6_duck_changed = None
# Chuyển lỗi ra thông báo dễ đọc, bảo đảm chạy phần dọn dẹp nếu có.
try:
    # Giá trị trung gian `q6_manual_setting` được dùng ngay ở bước bên dưới.
    q6_manual_setting = con.execute("SELECT current_setting('disabled_optimizers')").fetchone()[0]
    # Chọn nhánh phù hợp với dữ liệu hoặc trạng thái vừa đọc.
    if q6_manual_setting != "filter_pushdown":
        # In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
        print("Chưa tắt filter_pushdown. Sửa ô cấu hình ở mục 1.7 rồi chạy lại hai ô này.")
    else:
        # Chạy đúng một lượt Q6; EXPLAIN và JSON profiling dùng cùng cách với lượt gốc.
        q6_duck_changed = profile_duckdb_q6(con, q6_sql)
        # Hiển thị dữ liệu hoặc ảnh đã chuẩn bị trong output notebook.
        display(q6_control_result(q6_duck_default, q6_duck_changed, q6_df_value))
finally:
    # Trả về mặc định cả khi truy vấn lỗi hoặc bị ngắt bằng KeyboardInterrupt.
    con.execute("SET disabled_optimizers=''")
    # In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
    print("DuckDB đã trở về cấu hình mặc định; Q17 không dùng thiết lập đối chứng Q6.")


### 1.8. Kết luận Q6 theo số liệu vừa chạy

Đọc ba bằng chứng theo thứ tự:

1. **Kế hoạch:** DuckDB mặc định lọc trong `READ_PARQUET`; sau thay đổi phải thấy `FILTER` riêng giữa đọc và tính tích/tổng.
2. **Hồ sơ thực thi:** xem thời gian đường đọc/lọc trước và sau. Không lấy số toán tử ít hơn để suy ra nhanh hơn.
3. **Đối chứng:** kết quả phải giữ nguyên; bảng tự tính thời gian giảm hoặc tăng. Nếu không nhanh hơn trong lần chạy này, ghi nhận đúng số đo và không kết luận đã tối ưu thành công.

Nếu cả kế hoạch và thời gian thay đổi theo hướng thuận lợi, có bằng chứng rằng **đường lọc tích hợp của DuckDB 0.9.1 góp phần làm Q6 tốn thời gian trên bộ dữ liệu và môi trường này**. Chưa quy toàn bộ khoảng chênh lệch với DataFusion cho một dòng mã, ngôn ngữ lập trình hay việc đóng gói bit.

Cả hai lượt DuckDB đều có JSON profiling, mỗi cấu hình một lượt, mặc định chạy trước; không xóa bộ nhớ đệm. Phần chẩn đoán DataFusion là lượt chạy riêng và không dùng thay thời gian SELECT gốc. Bài báo trình bày kiến trúc và hiệu năng; đối chứng thay cấu hình Q6 ở đây do nhóm bổ sung, không gán kết luận này cho tác giả.

**Trình tự kiểm tra:** chạy Q6 với cấu hình mặc định → xem kế hoạch và hồ sơ thời gian → đối chiếu mã nguồn → tắt quy tắc `filter_pushdown` của DuckDB → chạy lại → so sánh kế hoạch, thời gian và kết quả.

## 2. Q17 — truy vấn con và phép nối


In [ ]:
# Hiển thị SQL có xuống dòng; câu thực thi vẫn lấy từ queries.sql.
q17_display = """
select
    sum(l_extendedprice) / 7.0 as avg_yearly
from lineitem, part
where p_partkey = l_partkey
  and p_brand = 'Brand#23'
  and p_container = 'MED BOX'
  and l_quantity < (
      select 0.2 * avg(l_quantity)
      from lineitem
      where l_partkey = p_partkey
  );
"""
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print(q17_display)


**Ý nghĩa truy vấn**

Q17 xét các mặt hàng có nhãn hiệu `Brand#23` và bao bì `MED BOX`. Điều kiện `p_partkey = l_partkey` ghép mỗi dòng hàng với thông tin mặt hàng tương ứng.

Truy vấn con tính `0.2 * AVG(l_quantity)` trên các dòng hàng của mặt hàng đang xét. `l_partkey` bên trong thuộc bảng `lineitem` của truy vấn con; `p_partkey` tham chiếu bảng `part` ở truy vấn ngoài.

Truy vấn ngoài chỉ giữ dòng có số lượng nhỏ hơn ngưỡng trên. Ví dụ, số lượng trung bình bằng 10 tạo ngưỡng bằng 2; dòng có số lượng 1 được giữ, còn dòng có số lượng 2 bị loại.

Kết quả là `SUM(l_extendedprice) / 7.0`, mang tên `avg_yearly`. Công thức không nhân thêm số lượng và không trừ giảm giá. Câu SQL không đặt điều kiện ngày. Bộ tối ưu có thể chuyển truy vấn con thành phép tổng hợp kết hợp với phép nối.

**Ý nghĩa trong hoạt động bán hàng**

Một doanh nghiệp muốn tập trung vào các lô hàng lớn để giảm chi phí xử lý những yêu cầu mua số lượng nhỏ. Doanh nghiệp đặt câu hỏi: **Nếu ngừng nhận các dòng hàng có số lượng rất nhỏ của những mặt hàng thuộc nhãn hiệu `Brand#23`, bao bì `MED BOX`, thì trung bình mỗi năm sẽ mất bao nhiêu doanh thu trước giảm giá?** Đây là mục đích nghiệp vụ của Q17 trong đặc tả TPC-H.

Ở đây, số lượng rất nhỏ được xác định bằng tiêu chí **nhỏ hơn 20% số lượng trung bình của từng mặt hàng**, không phải nhỏ hơn 20 sản phẩm hoặc lấy 20% số dòng bán hàng. Mỗi mặt hàng có ngưỡng riêng; số trung bình được tính trên toàn bộ các dòng của chính mặt hàng đó trong dữ liệu, kể cả các đơn đã thực hiện và đang chờ theo mô tả của TPC-H.

Ví dụ minh họa bằng đồng: giả sử một mặt hàng thuộc đúng nhãn hiệu và bao bì trên có ba dòng hàng trong toàn bộ dữ liệu bảy năm:

| Dòng hàng | Số lượng | Giá trị cả dòng trước giảm | Có được cộng vào Q17? |
|---|---:|---:|---|
| A | 1 | 70.000 đồng | Có, vì 1 < 2 |
| B | 9 | 630.000 đồng | Không, vì 9 không nhỏ hơn 2 |
| C | 20 | 1.400.000 đồng | Không, vì 20 không nhỏ hơn 2 |

Số lượng trung bình là `(1 + 9 + 20) / 3 = 10`; ngưỡng chọn là `0,2 × 10 = 2`. Chỉ dòng A được giữ. Giá trị trước giảm của các dòng đủ điều kiện là 70.000 đồng; chia cho 7 cho kết quả **10.000 đồng mỗi năm**. Nếu đây là mặt hàng duy nhất thỏa điều kiện nhãn hiệu và bao bì, đó chính là kết quả Q17 của ví dụ.

Con số này là mức doanh thu trước giảm giá trung bình hằng năm có thể mất nếu không nhận những dòng hàng nhỏ đã xác định. Doanh nghiệp có thể dùng nó làm một đầu vào khi cân nhắc lợi ích giảm chi phí xử lý. Truy vấn không tính chi phí tiết kiệm, không tính lợi nhuận và không đủ để kết luận nên ngừng bán các lô nhỏ. Nó cũng không loại toàn bộ một đơn hàng chỉ vì đơn đó có một dòng nhỏ.

Hệ số `7.0` phản ánh khoảng dữ liệu bảy năm trong thiết kế TPC-H; câu SQL không tự lọc bảy năm gần nhất và không tính riêng kết quả từng năm. Ví dụ bằng đồng chỉ dùng để minh họa, không xác định đơn vị tiền của dữ liệu TPC-H.

*Nguồn: Transaction Processing Performance Council, TPC Benchmark H, phiên bản 3.0.1, mục 2.4.17–2.4.17.1, tr. 57. [Đặc tả TPC-H](https://www.tpc.org/tpc_documents_current_versions/pdf/tpc-h_v3.0.1.pdf).*


### 2.1. Q17 gốc trên hai bộ máy

Trước khi lấy mốc Q17, đặt DataFusion về cấu hình lọc Parquet mặc định và DuckDB về danh sách quy tắc mặc định. Cùng dùng các tệp Parquet SF=10 đã đăng ký; một phân vùng DataFusion và một luồng DuckDB. Các ô dưới thực thi Q17 gốc một lần trên mỗi bộ máy.


In [ ]:
# Trả các lựa chọn đã thử trong Q6 về mặc định trước khi đo Q17.
ctx.sql("SET datafusion.execution.parquet.pushdown_filters = false").collect()
# Áp cấu hình hoặc câu lệnh DuckDB tại đúng bước được mô tả ở trên.
con.execute("SET disabled_optimizers=''")
# Chưa có số đo các phương án bổ sung cho lần chạy hiện tại.
q17_swapped_seconds = None
# Lưu kết quả Rust khi chỉ có các quy tắc chuẩn.
q17_rust_off = None
# Lưu kết quả Rust khi đăng ký quy tắc thống kê bổ sung.
q17_rust_on = None


### Thực thi Q17 bằng DataFusion


In [ ]:
# Lấy mốc bắt đầu đo Q17 bằng DataFusion.
start = perf_counter()
# Lập kế hoạch, thực thi Q17 và thu các lô kết quả.
q17_df_batches = ctx.sql(q17_sql).collect()
# Tính tổng thời gian hai bước gọi ở dòng trên.
q17_df_seconds = perf_counter() - start
# Q17 trả một giá trị; lấy giá trị Arrow đó sang Python.
q17_df_value = q17_df_batches[0].column(0)[0].as_py()
# In kết quả đã có trong bộ nhớ và thời gian đo được.
print("DataFusion — Q17:", q17_df_value)
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print(f"Thời gian một lượt: {q17_df_seconds:.3f} giây")


**Kế hoạch DataFusion Q17**

Nhánh `part` lọc nhãn hiệu và bao bì. Một nhánh `lineitem` phục vụ tính trung bình theo `l_partkey`; các phép nối kết hợp các kết quả trung gian trước phép tổng hợp cuối.

Trong DataFusion 32, phía trái của `HashJoinExec` là phía dựng bảng băm. Lựa chọn đầu vào cho phía này ảnh hưởng đến lượng dữ liệu lưu trong bộ nhớ và chi phí xây dựng cấu trúc tra cứu.

### Thực thi Q17 bằng DuckDB


In [ ]:
# Lấy mốc bắt đầu đo cùng Q17 bằng DuckDB.
start = perf_counter()
# Thực thi SQL và thu danh sách các dòng.
q17_duck_rows = con.execute(q17_sql).fetchall()
# Tính thời gian một lượt.
q17_duck_seconds = perf_counter() - start
# Lấy giá trị của dòng đầu, cột đầu.
q17_duck_value = q17_duck_rows[0][0]
# In giá trị và thời gian đã thu.
print("DuckDB — Q17:", q17_duck_value)
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print(f"Thời gian một lượt: {q17_duck_seconds:.3f} giây")
# Số thực có thể khác một lượng nhỏ do làm tròn; cho phép sai số tương đối 1e-12 hoặc tuyệt đối 1e-6.
print("Hai kết quả gần bằng nhau:", math.isclose(q17_df_value, q17_duck_value, rel_tol=1e-12, abs_tol=1e-6))


In [ ]:
# EXPLAIN hai bộ máy và hiển thị trong một bảng; không chạy lại Q17 để lấy kết quả.
# Bấm mở các mục bên dưới bảng để xem kế hoạch đầy đủ và các bước tối ưu.
display(compare_plans(ctx, con, q17_sql, "Q17", data_dir=data_dir))


### 2.2. Khoanh vùng vấn đề: phía dựng bảng băm

Phép nối mục tiêu là `lineitem.l_partkey = part.p_partkey`, với `part` đã lọc `Brand#23` và `MED BOX`. DataFusion 32 dựng bảng băm từ nhánh **trái** của `HashJoinExec`; DuckDB 0.9.1 dựng từ nhánh **phải** của `HASH_JOIN`.

Trong đợt khảo sát SF=10, DataFusion gốc dựng từ 59.986.052 dòng lineitem; DuckDB dựng từ 2.044 dòng part sau lọc. Cả hai vẫn phải đọc lineitem. Sau phép nối mục tiêu có 61.385 dòng; sau điều kiện so với trung bình còn 5.526 dòng.

`collect_left_input` của DataFusion thu và giữ các lô phía dựng, cấp `JoinHashMap::with_capacity(num_rows)`, tính băm rồi ghép các lô bằng `concat_batches`. Chọn đầu vào lớn làm tăng quy mô các công việc này. `build_time` có cả đọc đầu vào; `build_mem_used` là bộ đếm tài nguyên có phần ước lượng, không phải RAM đỉnh của tiến trình.

Nguồn: [HashJoinExec và phía dựng](https://github.com/apache/datafusion/blob/32.0.0/datafusion/physical-plan/src/joins/hash_join.rs#L168-L176), [collect_left_input](https://github.com/apache/datafusion/blob/32.0.0/datafusion/physical-plan/src/joins/hash_join.rs#L534-L599), [DuckDB: pipeline dựng bên phải](https://github.com/duckdb/duckdb/blob/v0.9.1/src/execution/operator/join/physical_join.cpp#L42-L47).


In [ ]:
# Hiển thị ảnh chú giải; không chạy truy vấn.
display(Image(filename=str(tpch_dir / "q17-assets" / "01-q17-phia-dung.png"), width=1100))


**Vì sao DataFusion không tự đổi phía?** Với cấu hình benchmark, `collect_statistics=false`. Các đầu vào thiếu thống kê cần để so kích thước; `should_swap_join_order` trả `false` khi không có đủ hai giá trị nên giữ nguyên hai phía. Không có nghĩa DataFusion không có bộ chọn phía dựng. Bật thu thống kê chưa đủ trong kiểm tra này vì nhánh part sau FilterExec vẫn thiếu thống kê.

Ảnh sau kết xuất nguyên văn mã DataFusion 32.0.0; vùng màu do nhóm bổ sung. Các đoạn nguồn và commit được ghi trong [source-snippets.json](q17-assets/source-snippets.json).


In [ ]:
# Hiển thị ảnh chú giải; không chạy truy vấn.
display(Image(filename=str(tpch_dir / "q17-assets" / "02-datafusion-chon-phia.png"), width=1100))


### 2.3. Đối chứng thứ nhất: đổi thứ tự hai bảng trong SQL

Chỉ thay `FROM lineitem, part` ở truy vấn ngoài bằng `FROM part, lineitem`; giữ nguyên truy vấn con, điều kiện và phép tính. Với DataFusion 32 và kế hoạch hiện tại, thay đổi này chuyển part sau lọc sang nhánh trái để dựng bảng băm.

Đây là đối chứng bằng SQL để hiểu vấn đề; chưa sử dụng quy tắc vật lý tự viết. Cần kiểm tra kế hoạch và kết quả sau thay đổi, vì bộ tối ưu ở phiên bản khác có thể tự đổi thứ tự tiếp.


In [ ]:
# Thay đúng một FROM ở truy vấn ngoài; dừng nếu không khớp mẫu mong đợi.
q17_swapped_sql, replacements = re.subn(
    r"from\s+lineitem\s*,\s*part", "from part, lineitem",
    q17_sql, count=1, flags=re.IGNORECASE,
)
# Chọn nhánh phù hợp với dữ liệu hoặc trạng thái vừa đọc.
if replacements != 1:
    # Dừng với thông báo rõ ràng thay vì tiếp tục bằng dữ liệu hoặc cấu hình không hợp lệ.
    raise ValueError("Không tìm thấy FROM lineitem, part của Q17.")
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print(q17_swapped_sql)


In [ ]:
# Chạy Q17 đã đổi FROM đúng một lần và thu kết quả.
start = perf_counter()
# Giá trị trung gian `q17_swapped_batches` được dùng ngay ở bước bên dưới.
q17_swapped_batches = ctx.sql(q17_swapped_sql).collect()
# Lưu thời gian đối chứng đổi FROM, chưa phải quy tắc vật lý.
q17_swapped_seconds = perf_counter() - start
# Giá trị trung gian `q17_swapped_value` được dùng ngay ở bước bên dưới.
q17_swapped_value = q17_swapped_batches[0].column(0)[0].as_py()
# Kiểm tra trường hợp thiếu dữ liệu hoặc chưa có giá trị trước khi dùng tiếp.
if not math.isclose(q17_swapped_value, q17_df_value, rel_tol=1e-12, abs_tol=1e-6):
    # Dừng với thông báo rõ ràng thay vì tiếp tục bằng dữ liệu hoặc cấu hình không hợp lệ.
    raise ValueError("Kết quả Q17 sau đổi FROM không tương đương kết quả gốc.")
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print("Kết quả:", q17_swapped_value)
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print(f"DataFusion — Q17 gốc:       {q17_df_seconds:.3f} giây")
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print(f"DataFusion — Q17 đổi FROM:  {q17_swapped_seconds:.3f} giây")
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print(f"DuckDB — Q17 gốc:           {q17_duck_seconds:.3f} giây")
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print("Kết quả giữ nguyên; mỗi số là một lượt đo, không xóa cache.")


In [ ]:
# EXPLAIN chỉ lập kế hoạch; không chạy lại Q17.
q17_swapped_explain = [row for batch in ctx.sql("EXPLAIN " + q17_swapped_sql).collect() for row in batch.to_pylist()]
# Duyệt từng mục theo thứ tự để thu dữ liệu hoặc ghép phần trình bày tương ứng.
for row in q17_swapped_explain:
    # Chọn nhánh phù hợp với dữ liệu hoặc trạng thái vừa đọc.
    if row["plan_type"] == "physical_plan":
        # In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
        print(row["plan"].replace(str(data_dir), "[DỮ LIỆU]").replace(str(data_dir).lstrip("/"), "[DỮ LIỆU]"))


### 2.4. Giữ SQL gốc: chọn phía dựng từ thống kê trước thực thi

Chương trình Rust dùng DataFusion **32.0.0** và đọc nguyên văn Q17 từ `queries.sql`. Hai phương án Rust đều bật `collect_statistics=true` trước khi đăng ký bảng; chỉ phương án bật đăng ký `StatisticsBuildSideRule` qua `add_physical_optimizer_rule`. Đây là đối chứng có thêm thu thống kê, khác cấu hình Python benchmark ban đầu.

**Căn cứ quyết định:** metadata Parquet cung cấp số dòng nguồn. `Filter` không tạo thêm dòng nên số dòng nguồn là giới hạn trên của đầu ra, ngay cả khi `FilterExec.statistics()` không trả ước lượng. `Projection` và `CoalesceBatches` giữ số dòng. Các toán tử chưa hỗ trợ không được tự suy diễn.

| Thông tin | Cách hiểu |
|---|---|
| `lower_rows`, `upper_rows` | Giới hạn số dòng có căn cứ từ nguồn và quy tắc truyền; không phải số đếm sau chạy. |
| `estimated_rows` | Ước lượng có sẵn nếu lấy được; `null` nghĩa là chưa biết, không phải bằng 0. |
| Chi phí dựng mô hình | Số dòng × (tổng độ rộng cột cố định + 32 byte chi phí băm mô hình mỗi dòng). |
| Quyết định đổi | Chi phí mô hình tối đa bên phải nhỏ hơn chi phí mô hình tối thiểu bên trái. |

Với Q17, nhánh lineitem không lọc có số dòng nguồn chính xác; nhánh part đã lọc có giới hạn trên từ số dòng nguồn. Quy tắc không chứa tên bảng, khóa riêng của Q17, nhãn hiệu, bao bì hoặc số dòng thực tế 2.044. Không chạy COUNT, lấy mẫu hay lấy kết quả truy vấn để chọn phía dựng.

32 byte là tham số chung của mô hình; không phải bộ nhớ đo được. Chưa mô hình hóa đầy đủ bộ cấp phát, bộ nhớ đệm, phân bố khóa hoặc chi phí CPU. Chỉ hỗ trợ INNER/CollectLeft không có join filter và hai đầu vào một phân vùng; kiểu cột có độ rộng thay đổi hoặc khoảng chi phí không đủ phân biệt thì giữ kế hoạch. Đây là quy tắc dựa trên thống kê và giới hạn, chưa phải bộ tối ưu chi phí đầy đủ hay thuật toán dự đoán chính xác mọi truy vấn.

Nguồn: [FilterExec.statistics](https://github.com/apache/datafusion/blob/32.0.0/datafusion/physical-plan/src/filter.rs#L201-L238), [PhysicalOptimizerRule](https://github.com/apache/datafusion/blob/32.0.0/datafusion/core/src/physical_optimizer/optimizer.rs#L34-L52), [đăng ký quy tắc](https://github.com/apache/datafusion/blob/32.0.0/datafusion/core/src/execution/context.rs#L1654-L1661). Phần suy luận và mô hình do nhóm bổ sung; thao tác đổi phía và phục hồi cột tham khảo [swap_hash_join của DataFusion 32](https://github.com/apache/datafusion/blob/32.0.0/datafusion/core/src/physical_optimizer/join_selection.rs#L130-L196).


In [ ]:
# Sơ đồ quy tắc thống kê, không chứa kết quả đo sau chạy.
display(Image(filename=str(tpch_dir / "q17-assets" / "08-q17-thong-ke-truoc-chay.png"), width=1100))


### 2.5. Mã thu thông tin, ra quyết định và sửa kế hoạch

- [statistics.rs](q17-physical-rule/src/statistics.rs): lấy số dòng nguồn, truyền giới hạn qua Filter/Projection/Coalesce và tính chi phí mô hình.
- [q17_rule.rs](q17-physical-rule/src/q17_rule.rs): kiểm tra phép nối hỗ trợ, so khoảng chi phí, ghi nhật ký rồi đổi đầu vào và phục hồi lược đồ.
- [main.rs](q17-physical-rule/src/main.rs): bật thu thống kê cho cả hai phương án, đăng ký quy tắc khi bật, chốt nhật ký trước `collect`.

Các mục mở rộng dưới đây đọc trực tiếp mã hiện tại. Sau khi sửa Rust, build lại ở 2.6 rồi chạy từ 2.7. Không bỏ Projection phục hồi cột vì toán tử phía trên vẫn tham chiếu thứ tự cột cũ.


In [ ]:
# Đọc trực tiếp mã hiện tại; mở mục tương ứng để xem chi tiết.
q17_project = tpch_dir / "q17-physical-rule"
# Duyệt từng mục theo thứ tự để thu dữ liệu hoặc ghép phần trình bày tương ứng.
for source_name, title in [
    ("statistics.rs", "1. Thu và truyền thống kê; tính chi phí mô hình"),
    ("q17_rule.rs", "2. So chi phí và biến đổi ExecutionPlan"),
    ("main.rs", "3. Đăng ký quy tắc và chốt quyết định trước collect"),
]:
    # Giá trị trung gian `source_path` được dùng ngay ở bước bên dưới.
    source_path = q17_project / "src" / source_name
    # Đọc nội dung tệp hiện tại để trình bày hoặc kiểm tra, không lấy từ output cũ.
    source_html = Code(source_path.read_text(), language="rust")._repr_html_()
    # Hiển thị dữ liệu hoặc ảnh đã chuẩn bị trong output notebook.
    display(HTML(f"<details><summary>{title}</summary>{source_html}</details>"))


### 2.6. Biên dịch chương trình Rust

Cargo tải và biên dịch các thư viện ở lần đầu; các lần sau dùng lại kết quả build. `--release` bật tối ưu khi biên dịch; `--locked` giữ bộ phiên bản trong Cargo.lock. DataFusion được ghim chính xác 32.0.0, Arrow/Parquet dòng 47; phụ thuộc bắc cầu và trình biên dịch của bản build này có thể khác wheel Python của tác giả.

Thư mục `target/` được bỏ qua bởi Git. Không đổi các gói Python hoặc sửa mã thư viện upstream. Thời gian build không đưa vào thời gian truy vấn.


In [ ]:
# Build lại sau mỗi lần sửa Rust; Cargo chỉ biên dịch các phần đã thay đổi.
import shutil
# Gọi chương trình ngoài bằng danh sách tham số và kiểm tra trạng thái.
import subprocess
# Kiểm tra trường hợp thiếu dữ liệu hoặc chưa có giá trị trước khi dùng tiếp.
if shutil.which("cargo") is None:
    # Dừng với thông báo rõ ràng thay vì tiếp tục bằng dữ liệu hoặc cấu hình không hợp lệ.
    raise RuntimeError("Chưa tìm thấy Cargo trong PATH của kernel; cần cài hoặc mở lại kernel sau khi cài Rust.")
# Lưu thư mục dự án Rust cần build hoặc đọc mã.
q17_project = tpch_dir / "q17-physical-rule"
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print("Đang build DataFusion Rust 32.0.0; lần đầu có thể cần vài phút.", flush=True)
# Chuyển lỗi ra thông báo dễ đọc, bảo đảm chạy phần dọn dẹp nếu có.
try:
    # Lưu đầu ra tiến trình Cargo, cần kiểm tra mã kết thúc.
    q17_build = subprocess.run(
        ["cargo", "build", "--release", "--locked", "-j", "3"],
        cwd=q17_project, capture_output=True, text=True, timeout=900,
    )
except subprocess.TimeoutExpired as error:
    # Dừng với thông báo rõ ràng thay vì tiếp tục bằng dữ liệu hoặc cấu hình không hợp lệ.
    raise RuntimeError("Build vượt 15 phút; lệnh Cargo đã bị dừng. Kiểm tra đầu ra build trước khi thử lại.") from error
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print(q17_build.stdout)
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print(q17_build.stderr)
# Báo lỗi nếu tiến trình build không thành công.
q17_build.check_returncode()
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print("Đã build. Tệp chạy:", q17_project / "target" / "release" / "q17-physical-rule")


### 2.7. Xem căn cứ và kế hoạch trước khi chạy Q17

Ô dưới dùng `--plan-only`: đăng ký bảng, đọc thống kê metadata và lập kế hoạch, chưa gọi `collect` của Q17. Bảng giải thích vì sao đổi hoặc giữ phía dựng; các bộ đếm thực thi chưa được dùng.

Hình khoanh **5–6–9** chỉ rõ cây ExecutionPlan, vòng tối ưu vật lý và ô xanh dương nơi quy tắc bổ sung can thiệp. Số liệu trong bảng được chốt trước collect; phần tạo hình không chạy thêm truy vấn.


In [ ]:
# Chỉ lập kế hoạch: kết quả Q17 chưa được tính.
q17_pre_execution = run_q17_rule(root, enabled=True, plan_only=True)
# Hiển thị dữ liệu hoặc ảnh đã chuẩn bị trong output notebook.
display(show_estimation_decisions(q17_pre_execution))
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print(q17_pre_execution["physical_plan"].replace(str(data_dir), "[DỮ LIỆU]").replace(str(data_dir).lstrip("/"), "[DỮ LIỆU]"))


### 2.8. Chạy Q17 gốc: thu thống kê, chưa đăng ký quy tắc riêng

Chạy một lần bằng binary Rust đã build. Thu thống kê nguồn bật ở cả hai phương án. Thời gian chuẩn bị có đăng ký bảng/metadata được báo riêng; thời gian lập kế hoạch và thực thi được báo riêng. Không so trực tiếp thời gian Rust với wheel Python để quy lợi ích cho quy tắc.


In [ ]:
# Cùng collect_statistics=true như phương án bật; chưa thêm quy tắc riêng.
q17_rust_off = run_q17_rule(root, enabled=False)
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print("Kết quả:", q17_rust_off["value"])
# In trạng thái hoặc kết quả đã thu, không chạy lại SQL để in.
print(f"Chuẩn bị: {q17_rust_off['preparation_seconds']:.3f}s; lập kế hoạch + chạy: {q17_rust_off['total_seconds']:.3f}s")


### 2.9. Chạy Q17 gốc: bật quy tắc thống kê và so sánh

Lập kế hoạch từ cùng SQL và cùng nguồn dữ liệu, sau đó chạy một lần. Bảng hiển thị quyết định trước thực thi cùng bộ đếm thực tế sau thực thi. Không bắt buộc phải đổi đúng một lần: nếu không đủ căn cứ thì quy tắc giữ kế hoạch và bảng phải phản ánh đúng điều đó.


In [ ]:
# Chạy một lượt Q17 gốc với quy tắc thống kê; không sửa SQL.
q17_rust_on = run_q17_rule(root, enabled=True)
# Hiển thị dữ liệu hoặc ảnh đã chuẩn bị trong output notebook.
display(show_rule_comparison(q17_rust_off, q17_rust_on, q17_df_value, data_dir=data_dir))


### 2.10. Đọc kết quả và phạm vi kết luận

- **Đổi FROM ở 2.3:** đối chứng bằng SQL để hiểu tác động của phía dựng.
- **Quy tắc ở 2.4–2.9:** giữ SQL gốc, dùng metadata và giới hạn số dòng để quyết định trước thực thi tại điểm mở rộng màu xanh dương.

Đọc theo thứ tự: (1) số dòng nguồn và giới hạn sau lọc; (2) độ rộng cột và chi phí mô hình; (3) quyết định đổi/giữ trước collect; (4) cây vật lý; (5) kết quả và thời gian thực tế. Số 2.044 nếu xuất hiện trong bộ đếm sau chạy là kết quả quan sát, không phải đầu vào của bộ chọn.

So hiệu quả bằng cặp Rust tắt/bật cùng binary và cùng collect_statistics=true. Xem cả chuẩn bị, lập kế hoạch và thực thi; không đưa build vào thời gian truy vấn. Mỗi cấu hình một lượt, không xóa cache, không cam kết nhanh hơn DuckDB trong mọi lần chạy. Nếu không đổi hoặc thời gian tăng, ghi nhận đúng kết quả.

Nhánh AVG vẫn tồn tại; không coi toàn bộ Q17 là thời gian dựng bảng băm. Quy tắc không sửa bản DataFusion upstream hoặc nâng phiên bản. Nó bổ sung một chính sách thống kê có phạm vi hỗ trợ rõ ràng qua PhysicalOptimizerRule, chưa thay toàn bộ bộ ước lượng hay bộ tối ưu chi phí của DataFusion.

## 3. Thời gian thực thi

Các số ở ô sau là lượt Python gốc, không bị thay bằng số Rust. Phương án Rust thống kê được so trong bảng riêng phía trên.


In [ ]:
# Chỉ in lại thời gian đã lưu; không chạy truy vấn lần nữa.
print(f"Q6  | DataFusion: {q6_df_seconds:.3f}s | DuckDB: {q6_duck_seconds:.3f}s")
# Mỗi số là một lượt đo, không phải trung bình.
print(f"Q17 | DataFusion: {q17_df_seconds:.3f}s | DuckDB: {q17_duck_seconds:.3f}s")


Nguồn: [bài báo DataFusion SIGMOD Companion 2024](https://doi.org/10.1145/3626246.3653368), Hình 2, mục 5 và 8; [mã benchmark](https://github.com/JayjeetAtGithub/datafusion-duckdb-benchmark); [HashJoinExec DataFusion 32](https://github.com/apache/datafusion/blob/32.0.0/datafusion/physical-plan/src/joins/hash_join.rs); [hash join DuckDB 0.9.1](https://github.com/duckdb/duckdb/blob/v0.9.1/src/execution/operator/join/physical_hash_join.cpp).


In [ ]:
# Đóng cơ sở dữ liệu DuckDB tạm trong bộ nhớ khi xem xong.
con.close()
# Biến con còn tên nhưng kết nối đã đóng; tạo lại bằng ô đăng ký bảng nếu muốn chạy tiếp.
print("Đã đóng DuckDB. Muốn chạy tiếp: chạy lại ô đăng ký bảng trước.")
